In [1]:
# Cell 1: Install dependencies
!pip install datasets huggingface_hub timm albumentations -q

import os, random, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2
from datasets import load_dataset
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cpu


In [4]:
# Cell 2: Load dataset from HuggingFace
ds = load_dataset("timm/oxford-iiit-pet")
print(ds)
print(ds['train'].features)

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'image_id', 'label_cat_dog'],
        num_rows: 3680
    })
    test: Dataset({
        features: ['image', 'label', 'image_id', 'label_cat_dog'],
        num_rows: 3669
    })
})
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['abyssinian', 'american_bulldog', 'american_pit_bull_terrier', 'basset_hound', 'beagle', 'bengal', 'birman', 'bombay', 'boxer', 'british_shorthair', 'chihuahua', 'egyptian_mau', 'english_cocker_spaniel', 'english_setter', 'german_shorthaired', 'great_pyrenees', 'havanese', 'japanese_chin', 'keeshond', 'leonberger', 'maine_coon', 'miniature_pinscher', 'newfoundland', 'persian', 'pomeranian', 'pug', 'ragdoll', 'russian_blue', 'saint_bernard', 'samoyed', 'scottish_terrier', 'shiba_inu', 'siamese', 'sphynx', 'staffordshire_bull_terrier', 'wheaten_terrier', 'yorkshire_terrier']), 'image_id': Value('string'), 'label_cat_dog': ClassLabel(names=['cat', 'dog'])}


In [5]:
# Cell 3 (UPDATED): Basic EDA
train_ds = ds['train']
test_ds  = ds['test']

print(f"Train samples: {len(train_ds)}")
print(f"Test  samples: {len(test_ds)}")

# Inspect one sample
sample = train_ds[0]
print("Keys:", sample.keys())
print("Image size:", sample['image'].size)
print("Label (breed):", sample['label'])
print("Label (cat/dog):", sample['label_cat_dog'])   # 0=cat, 1=dog
print("Image ID:", sample['image_id'])

# Find the mask key
mask_key = None
for k in sample.keys():
    if 'mask' in k.lower() or 'seg' in k.lower() or 'trimap' in k.lower():
        mask_key = k
        break

print(f"\nDetected mask key: '{mask_key}'")
print(f"Mask type: {type(sample[mask_key])}")

# Unique breed labels
labels_list = [s['label'] for s in train_ds]
print(f"Unique breed labels: {len(set(labels_list))}")  # should be 37

Train samples: 3680
Test  samples: 3669
Keys: dict_keys(['image', 'label', 'image_id', 'label_cat_dog'])
Image size: (389, 500)
Label (breed): 20
Label (cat/dog): 0
Image ID: Maine_Coon_204

Detected mask key: 'None'


KeyError: None